# Embedding Server — Colab Test

Tests the queue-based embedding server with live + bulk endpoints.

**Runtime type:** GPU (L4 or better)

## 1. Install dependencies

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pyngrok httpx
!pip install -q "transformers>=4.52.0" accelerate torch numpy tqdm

# FP8 + flash-attn (L4 / Ada Lovelace)
!pip install -q transformer-engine
!pip install -q flash-attn --no-build-isolation

## 2. Upload project

Zip the `embeddings/` directory locally and upload when prompted:
```
# from the repo root:
zip -r embeddings.zip embeddings/ -x 'embeddings/.venv/*' 'embeddings/__pycache__/*'
```

In [ ]:
import os

if not os.path.exists("embeddings/server.py"):
    from google.colab import files
    uploaded = files.upload()          # select embeddings.zip
    !unzip -o embeddings.zip

os.chdir("embeddings")
print("Working dir:", os.getcwd())
!ls -la *.py modules/*.py

## 3. Authenticate ngrok

Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok, conf

# Uncomment and paste your token:
# conf.get_default().auth_token = "YOUR_NGROK_TOKEN_HERE"

# Or use Colab secrets:
# from google.colab import userdata
# conf.get_default().auth_token = userdata.get('NGROK_TOKEN')

## 4. Start server

In [ ]:
import subprocess, time, sys

# Kill any previous instance
!pkill -f 'python server.py' 2>/dev/null || true
time.sleep(1)

# Start server in background, capture output to log file
log = open("server.log", "w")
proc = subprocess.Popen(
    [sys.executable, "server.py"],
    stdout=log, stderr=subprocess.STDOUT,
)
print(f"Server PID: {proc.pid}")

# Wait for model to load (~15-30s on L4)
print("Waiting for model to load…")
for i in range(60):
    time.sleep(2)
    if proc.poll() is not None:
        print("Server crashed! Last 30 lines of log:")
        log.close()
        with open("server.log") as f:
            lines = f.readlines()
            for l in lines[-30:]:
                print(l, end="")
        raise RuntimeError("Server exited")
    # Check if server is ready
    import httpx
    try:
        r = httpx.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            print(f"Server ready after {i*2}s")
            print(r.json())
            break
    except httpx.ConnectError:
        pass
else:
    print("Timed out waiting for server")
    raise RuntimeError("Server did not start")

## 5. Expose via ngrok

In [ ]:
# Close any existing tunnels
ngrok.kill()

tunnel = ngrok.connect(8000)
PUBLIC_URL = str(tunnel.public_url)
print(f"Public URL: {PUBLIC_URL}")

## 6. Run tests

In [ ]:
import httpx, time, json

BASE = "http://localhost:8000"  # use PUBLIC_URL to test from outside

print("=== Health check ===")
r = httpx.get(f"{BASE}/health")
print(json.dumps(r.json(), indent=2))

print("\n=== Queue status (empty) ===")
r = httpx.get(f"{BASE}/queue")
print(json.dumps(r.json(), indent=2))

In [ ]:
print("=== Live embed ===")
r = httpx.post(f"{BASE}/embed", json={
    "documents": [
        {"doc_id": "test1", "text": "Hello world, this is a test document."},
        {"doc_id": "test2", "text": "Another document with different content."},
    ]
}, timeout=120)
print(f"Status: {r.status_code}")
data = r.json()
print(f"Results: {len(data['results'])} documents")
for doc in data['results']:
    print(f"  {doc['doc_id']}: {len(doc['embeddings'])} chunks, "
          f"embedding dim={len(doc['embeddings'][0]) if doc['embeddings'] else 0}")
print(f"Stats: {json.dumps(data['stats'], indent=2)}")

In [ ]:
print("=== Bulk embed ===")
docs = [{"doc_id": f"d{i}", "text": f"This is document number {i}. It contains some text for embedding."} for i in range(10)]
r = httpx.post(f"{BASE}/embed/bulk", json={
    "documents": docs,
    "chunk_size": 3,
}, timeout=30)
print(f"Status: {r.status_code}")
job = r.json()
print(json.dumps(job, indent=2))
JOB_ID = job["job_id"]

In [ ]:
print("=== Poll bulk job ===")
for attempt in range(30):
    r = httpx.get(f"{BASE}/embed/bulk/{JOB_ID}")
    status = r.json()
    print(f"  [{attempt}] status={status['status']} "
          f"chunks={status.get('completed_chunks', '?')}/{status.get('total_chunks', '?')}")
    if status["status"] in ("completed", "failed"):
        break
    time.sleep(3)

if status["status"] == "completed":
    total_docs = sum(len(chunk) for chunk in status["results"])
    print(f"\nCompleted: {total_docs} document results across {len(status['results'])} chunks")
    print(f"Stats: {json.dumps(status['stats'], indent=2)}")
else:
    print(f"\nJob failed: {status}")

In [ ]:
print("=== Queue status (after processing) ===")
r = httpx.get(f"{BASE}/queue")
print(json.dumps(r.json(), indent=2))

print("\n=== 404 for nonexistent job ===")
r = httpx.get(f"{BASE}/embed/bulk/deadbeef")
print(f"Status: {r.status_code}, body: {r.json()}")

print("\n=== Validation error (empty documents) ===")
r = httpx.post(f"{BASE}/embed", json={"documents": []})
print(f"Status: {r.status_code}")

print("\n=== Validation error (too many docs for live) ===")
r = httpx.post(f"{BASE}/embed", json={
    "documents": [{"doc_id": f"d{i}", "text": "x"} for i in range(101)]
})
print(f"Status: {r.status_code}")

## 7. Priority test: live request during bulk processing

Submits a bulk job, then immediately sends a live request.
The live request should complete even though bulk chunks are queued first.

In [ ]:
import threading

print("=== Priority test ===")

# Submit bulk (many small chunks)
docs = [{"doc_id": f"bulk_d{i}", "text": f"Bulk document {i}."} for i in range(20)]
r = httpx.post(f"{BASE}/embed/bulk", json={
    "documents": docs,
    "chunk_size": 2,  # 10 chunks
}, timeout=30)
bulk_job = r.json()
print(f"Bulk submitted: job_id={bulk_job['job_id']}, chunks={bulk_job['total_chunks']}")

# Immediately send live request
t0 = time.time()
r = httpx.post(f"{BASE}/embed", json={
    "documents": [{"doc_id": "live_priority", "text": "This live request should jump the queue."}]
}, timeout=120)
elapsed = time.time() - t0
print(f"Live request completed in {elapsed:.1f}s, status={r.status_code}")
assert r.status_code == 200, f"Live request failed: {r.text}"
print("PRIORITY TEST PASSED")

## 8. Cleanup

In [ ]:
# Stop server and close tunnel
proc.terminate()
ngrok.kill()
log.close()
print("Cleaned up")

# Show final server log
print("\n=== Server log (last 40 lines) ===")
with open("server.log") as f:
    lines = f.readlines()
    for l in lines[-40:]:
        print(l, end="")